In [0]:
%sql
-- Gold Validation: Aggregation Table Storage Location
-- Domain: Data Engineering
-- Purpose: Verify that the aggregation table is physically stored
--          in the Gold ADLS container

DESCRIBE DETAIL adbdevbankproject.gold.agg_transaction_daily;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,774c32e9-bacc-4d26-9610-16df0c52cca1,adbdevbankproject.gold.agg_transaction_daily,null,abfss://gold@stgdevbankproject.dfs.core.windows.net/__unitystorage/schemas/68ba34e6-1e50-4869-b4b3-81e03b1e22d4/tables/fadf3da7-58f4-4574-ad55-68b1572e9afb,2026-08-23T15:29:50.916Z,2026-08-23T15:29:54.000Z,List(),List(),1,3380265,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

SELECT
    COUNT(*) AS aggregation_rows,

    SUM(transaction_count) AS aggregated_transaction_count,

    SUM(total_amount) AS aggregated_total_amount

FROM adbdevbankproject.gold.agg_transaction_daily;

aggregation_rows,aggregated_transaction_count,aggregated_total_amount
487233,13305909,571835398.56


In [0]:
%sql

-- Gold Aggregation Layer: Daily Transaction Aggregation
-- Domain: Power BI Performance Optimization
-- Purpose: Pre-aggregate frequently used transaction metrics
--          to reduce the amount of detailed data queried by Power BI.
--
-- Grain:
-- One row per Date + Merchant Category + Merchant Country + Transaction Type
--
-- Dimensions:
-- date_key
-- mcc_key
-- merchant_country
-- transaction_type
--
-- Measures:
-- transaction_count
-- total_amount
--
-- Design Decision:
-- The aggregation grain was selected to serve the largest number
-- of common queries and core KPIs without using the detailed fact grain.
--
-- Average transaction amount is intentionally not stored.
-- It will be calculated as Total Amount / Transaction Count
-- in Power BI to maintain aggregation accuracy.

CREATE OR REPLACE TABLE adbdevbankproject.gold.agg_transaction_daily
AS

SELECT
    f.date_key,
    f.mcc_key,
    f.merchant_country,
    f.use_chip AS transaction_type,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount

FROM adbdevbankproject.gold.fact_transactions f

GROUP BY
    f.date_key,
    f.mcc_key,
    f.merchant_country,
    f.use_chip;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Gold Aggregation Layer: Transaction Aggregation Profiling
-- Domain: Power BI Performance Optimization
-- Purpose: Estimate the aggregation size before creating the physical aggregation table

SELECT
    COUNT(*) AS aggregated_row_count

FROM (
    SELECT
        f.date_key,
        f.mcc_key,
        f.merchant_country,
        f.use_chip

    FROM adbdevbankproject.gold.fact_transactions f

    GROUP BY
        f.date_key,
        f.mcc_key,
        f.merchant_country,
        f.use_chip
);

aggregated_row_count
487233


In [0]:
%sql
-- Gold Validation: Inspect Gold Dimension Storage Locations
-- Domain: Data Engineering
-- Purpose: Identify the physical storage locations of Gold dimension tables





DESCRIBE DETAIL adbdevbankproject.gold.dim_users;




format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,eed23266-b86a-4d7a-9529-fd4d06ec8c12,adbdevbankproject.gold.dim_users,null,abfss://gold@stgdevbankproject.dfs.core.windows.net/__unitystorage/schemas/68ba34e6-1e50-4869-b4b3-81e03b1e22d4/tables/ffa92374-a36e-4a1e-bb1f-23014db3cfe9,2026-08-22T14:22:17.420Z,2026-08-22T14:23:01.000Z,List(),List(),1,52256,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-0eadc4ef-f442-4a56-b12a-92136b0e7dec, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-ecff5ae6-dc88-4e16-b0be-784c6253e1f7)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Gold Validation: Fact-to-Dimension Referential Integrity
-- Domain: Data Quality
-- Purpose: Validate that all fact transactions have matching MCC and Date dimension records

SELECT
    'MCC' AS dimension_name,

    COUNT(*) AS total_fact_transactions,

    COUNT(m.mcc_key) AS matched_transactions,

    COUNT(*) - COUNT(m.mcc_key) AS unmatched_transactions

FROM adbdevbankproject.gold.fact_transactions f

LEFT JOIN adbdevbankproject.gold.dim_mcc m
    ON f.mcc_key = m.mcc_key

UNION ALL

SELECT
    'Date' AS dimension_name,

    COUNT(*) AS total_fact_transactions,

    COUNT(d.date_key) AS matched_transactions,

    COUNT(*) - COUNT(d.date_key) AS unmatched_transactions

FROM adbdevbankproject.gold.fact_transactions f

LEFT JOIN adbdevbankproject.gold.dim_date d
    ON f.date_key = d.date_key;

dimension_name,total_fact_transactions,matched_transactions,unmatched_transactions
MCC,13305909,13305909,0
Date,13305909,13305909,0


In [0]:
%sql
-- Gold Validation: Dimension Key Uniqueness
-- Domain: Data Quality
-- Purpose: Validate that surrogate keys are unique in all Gold dimension tables

SELECT
    'dim_users' AS dimension_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_key) AS distinct_keys,
    COUNT(*) - COUNT(DISTINCT user_key) AS duplicate_keys
FROM adbdevbankproject.gold.dim_users

UNION ALL

SELECT
    'dim_cards' AS dimension_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT card_key) AS distinct_keys,
    COUNT(*) - COUNT(DISTINCT card_key) AS duplicate_keys
FROM adbdevbankproject.gold.dim_cards

UNION ALL

SELECT
    'dim_mcc' AS dimension_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT mcc_key) AS distinct_keys,
    COUNT(*) - COUNT(DISTINCT mcc_key) AS duplicate_keys
FROM adbdevbankproject.gold.dim_mcc

UNION ALL

SELECT
    'dim_date' AS dimension_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date_key) AS distinct_keys,
    COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys
FROM adbdevbankproject.gold.dim_date;

dimension_name,total_rows,distinct_keys,duplicate_keys
dim_users,2000,2000,0
dim_cards,6146,6146,0
dim_mcc,109,109,0
dim_date,3591,3591,0


In [0]:
%sql
-- Gold Validation: Gold Layer Table Inventory
-- Domain: Data Quality
-- Purpose: Verify the final tables available in the Gold layer

SHOW TABLES IN adbdevbankproject.gold;

database,tableName,isTemporary
gold,dim_cards,false
gold,dim_date,false
gold,dim_mcc,false
gold,dim_users,false
gold,fact_transactions,false


In [0]:
%sql
-- Gold Analytics: KPI 06 - Transaction Performance by Transaction Type
-- Domain: Transaction Behavior Analysis
-- Purpose: Analyze transaction volume and transaction value by transaction type

SELECT
    f.use_chip AS transaction_type,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount,

    AVG(f.amount) AS average_transaction_amount

FROM adbdevbankproject.gold.fact_transactions f

GROUP BY
    f.use_chip

ORDER BY
    total_amount DESC;

transaction_type,transaction_count,total_amount,average_transaction_amount
Swipe Transaction,6967179,287151705.11,41.214917
Chip Transaction,4780818,195483583.94,40.889150
Online Transaction,1557912,89200109.51,57.256193


In [0]:
%sql
-- Gold Analytics: KPI 05 - Geographic Transaction Performance
-- Domain: Geographic Analysis
-- Purpose: تحليل حجم وقيمة المعاملات حسب دولة التاجر
--
-- المشكلة:
-- المعاملات الإلكترونية (Online Transactions) تظهر في طبقة Bronze
-- بقيمة merchant_city = 'ONLINE'، بينما merchant_state و zip فارغة (NULL).
-- كما أن المصدر لا يحتوي على أي معلومة جغرافية إضافية يمكن من خلالها
-- تحديد دولة التاجر لهذه المعاملات.
--
-- القرار:
-- الإبقاء على هذه المعاملات وعدم حذفها، مع الاحتفاظ بقيمة
-- merchant_country كـ NULL.
-- عدم استخدام 'Online Transactions' كقيمة داخل merchant_country،
-- لأن نوع المعاملة والموقع الجغرافي للتاجر بُعدان تحليليان مختلفان.
--
-- الحل:
-- استبعاد السجلات التي تحتوي على merchant_country = NULL من KPI
-- الخاص بالتحليل الجغرافي فقط.
-- تبقى هذه المعاملات موجودة في fact_transactions، ويمكن تحليلها
-- بشكل منفصل باستخدام use_chip لتحليل نوع المعاملة.

SELECT
    f.merchant_country,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount,

    AVG(f.amount) AS average_transaction_amount

FROM adbdevbankproject.gold.fact_transactions f

WHERE f.merchant_country IS NOT NULL

GROUP BY
    f.merchant_country

ORDER BY
    total_amount DESC;

merchant_country,transaction_count,total_amount,average_transaction_amount
United States,11653203,479603321.61,41.156352
Mexico,27401,889584.27,32.465394
Italy,7081,456054.18,64.405335
Canada,10647,358294.84,33.652187
United Kingdom,4482,169474.49,37.812247
Germany,3025,109560.05,36.218198
China,2908,108447.00,37.292641
France,2836,99789.70,35.186777
Japan,2107,82985.75,39.385738
Spain,1651,64457.98,39.041781


In [0]:
%sql
-- Gold Analytics: Data Quality Investigation - Online Transaction Geography
-- Domain: Geographic Analysis
-- Purpose: Identify available geographic attributes for online transactions

SELECT *
FROM adbdevbankproject.bronze.transactions_data
WHERE merchant_city = 'ONLINE'
LIMIT 10;

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
22730827,2019-03-27T14:46:00.000Z,1198,2804,$36.07,Online Transaction,41122,ONLINE,null,null,4784,null
22730829,2019-03-27T14:46:00.000Z,1330,4579,$39.04,Online Transaction,15143,ONLINE,null,null,4784,null
22730833,2019-03-27T14:47:00.000Z,1540,5562,$48.16,Online Transaction,39021,ONLINE,null,null,4784,null
22730849,2019-03-27T14:51:00.000Z,1138,5292,$56.31,Online Transaction,39021,ONLINE,null,null,4784,null
22730853,2019-03-27T14:52:00.000Z,3,4327,$68.02,Online Transaction,9932,ONLINE,null,null,5311,null
22730855,2019-03-27T14:52:00.000Z,1194,4614,$16.67,Online Transaction,39021,ONLINE,null,null,4784,null
22730856,2019-03-27T14:52:00.000Z,1361,4950,$46.53,Online Transaction,81759,ONLINE,null,null,7349,null
22730863,2019-03-27T14:54:00.000Z,147,2634,$138.02,Online Transaction,87530,ONLINE,null,null,4900,null
22730881,2019-03-27T14:59:00.000Z,1347,5678,$58.73,Online Transaction,9932,ONLINE,null,null,5311,null
22730907,2019-03-27T15:05:00.000Z,845,4599,$37.60,Online Transaction,73186,ONLINE,null,null,4814,null


In [0]:
%sql
-- Gold Analytics: Data Quality Check - Source Geography Validation
-- Domain: Geographic Analysis
-- Purpose: Verify the original geographic values before Silver transformation

SELECT
    merchant_city,
    merchant_state,
    COUNT(*) AS transaction_count

FROM adbdevbankproject.bronze.transactions_data

WHERE merchant_city IS NULL
   OR merchant_state IS NULL

GROUP BY
    merchant_city,
    merchant_state

ORDER BY
    transaction_count DESC;

merchant_city,merchant_state,transaction_count
ONLINE,null,1563700


In [0]:
%sql
-- Gold Analytics: Data Quality Check - Online Transactions with Missing Geography
-- Domain: Geographic Analysis
-- Purpose: Validate whether NULL geography originates from online transactions

SELECT
    merchant_city,
    merchant_state,
    merchant_country,
    COUNT(*) AS transaction_count

FROM adbdevbankproject.silver.transactions_data

WHERE merchant_country IS NULL

GROUP BY
    merchant_city,
    merchant_state,
    merchant_country

ORDER BY
    transaction_count DESC;

merchant_city,merchant_state,merchant_country,transaction_count
null,null,null,1563700


In [0]:
%sql
-- Gold Analytics: Data Quality Check - NULL Merchant Country Details
-- Domain: Geographic Analysis
-- Purpose: Inspect source geography fields for transactions with missing merchant country

SELECT
    merchant_state,
    merchant_city,
    COUNT(*) AS transaction_count

FROM adbdevbankproject.gold.fact_transactions

WHERE merchant_country IS NULL

GROUP BY
    merchant_state,
    merchant_city

ORDER BY
    transaction_count DESC;

merchant_state,merchant_city,transaction_count
null,null,1563700


In [0]:
%sql
-- Gold Analytics: Data Quality Check - NULL Merchant Country
-- Domain: Geographic Analysis
-- Purpose: Identify transactions with missing merchant country before KPI analysis

SELECT
    COUNT(*) AS null_country_transaction_count,
    SUM(amount) AS null_country_total_amount,
    AVG(amount) AS null_country_average_amount

FROM adbdevbankproject.gold.fact_transactions

WHERE merchant_country IS NULL;

null_country_transaction_count,null_country_total_amount,null_country_average_amount
1563700,88896665.09,56.850205


In [0]:
%sql
-- Gold Analytics: KPI 05 - Geographic Transaction Performance
-- Domain: Geographic Analysis
-- Purpose: Analyze transaction volume and transaction value by merchant country

SELECT
    f.merchant_country,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount,

    AVG(f.amount) AS average_transaction_amount

FROM adbdevbankproject.gold.fact_transactions f

GROUP BY
    f.merchant_country

ORDER BY
    total_amount DESC;

merchant_country,transaction_count,total_amount,average_transaction_amount
United States,11653203,479603321.61,41.156352
null,1563700,88896665.09,56.850205
Mexico,27401,889584.27,32.465394
Italy,7081,456054.18,64.405335
Canada,10647,358294.84,33.652187
United Kingdom,4482,169474.49,37.812247
Germany,3025,109560.05,36.218198
China,2908,108447.00,37.292641
France,2836,99789.70,35.186777
Japan,2107,82985.75,39.385738


In [0]:
%sql
-- Gold Analytics: KPI 04 - Customer Performance
-- Domain: Customer Analysis
-- Purpose: Analyze transaction volume and transaction value by customer

SELECT
    u.client_id,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount,

    AVG(f.amount) AS average_transaction_amount

FROM adbdevbankproject.gold.fact_transactions f

JOIN adbdevbankproject.gold.dim_users u
    ON f.user_key = u.user_key

GROUP BY
    u.client_id

ORDER BY
    total_amount DESC;

client_id,transaction_count,total_amount,average_transaction_amount
96,38617,2445773.25,63.334108
1686,19810,2167880.90,109.433665
1340,22023,2039921.23,92.626855
840,15095,1956340.84,129.601911
464,27619,1882901.35,68.174132
490,21831,1711482.69,78.396898
704,20748,1635022.05,78.803839
285,32032,1615458.99,50.432661
488,23990,1611114.42,67.157750
1168,30520,1590822.75,52.123943


In [0]:
%sql
-- Gold Analytics: KPI 03 - Card Performance
-- Domain: Card Analysis
-- Purpose: Analyze transaction volume and transaction value by card type and brand

SELECT
    c.card_brand,
    c.card_type,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount,

    AVG(f.amount) AS average_transaction_amount

FROM adbdevbankproject.gold.fact_transactions f

JOIN adbdevbankproject.gold.dim_cards c
    ON f.card_key = c.card_key

GROUP BY
    c.card_brand,
    c.card_type

ORDER BY
    total_amount DESC;

card_brand,card_type,transaction_count,total_amount,average_transaction_amount
Mastercard,Debit,5266958,207445238.14,39.386158
Visa,Debit,3014034,118646151.97,39.364570
Visa,Credit,1660478,93918588.14,56.561176
Mastercard,Credit,1257758,66551690.94,52.912954
Amex,Credit,854490,46943539.40,54.937494
Discover,Credit,336463,18398001.53,54.680608
Mastercard,Debit (Prepaid),632679,13927992.07,22.014311
Visa,Debit (Prepaid),283049,6004196.37,21.212569


In [0]:
%sql
-- Gold Analytics: KPI 02 - Merchant Category Performance
-- Domain: Merchant Analysis
-- Purpose: Analyze transaction volume and transaction value by merchant category

SELECT
    m.mcc_code,
    m.merchant_category,

    COUNT(*) AS transaction_count,

    SUM(f.amount) AS total_amount,

    AVG(f.amount) AS average_transaction_amount

FROM adbdevbankproject.gold.fact_transactions f

JOIN adbdevbankproject.gold.dim_mcc m
    ON f.mcc_key = m.mcc_key

GROUP BY
    m.mcc_code,
    m.merchant_category

ORDER BY
    total_amount DESC;

mcc_code,merchant_category,transaction_count,total_amount,average_transaction_amount
4829,Money Transfer,589140,53158515.64,90.230702
5411,"Grocery Stores, Supermarkets",1592578,40970630.43,25.725980
5300,Wholesale Clubs,601942,37697546.74,62.626543
5912,Drug Stores and Pharmacies,772913,35113527.69,45.430117
5541,Service Stations,1424711,29570426.66,20.755386
4900,"Utilities - Electric, Gas, Water, Sanitary",242993,27650038.08,113.789443
5311,Department Stores,475384,27031968.70,56.863438
5812,Eating Places and Restaurants,999738,26348225.47,26.355131
7538,Automotive Service Shops,478263,24955640.73,52.179744
4814,Telecommunication Services,218243,24726472.83,113.297897


In [0]:
%sql
-- Gold Analytics: KPI 01 - Monthly Transaction Performance
-- Domain: Transaction Trends
-- Purpose: Analyze monthly transaction volume and transaction value over time

SELECT
    d.year_number,
    d.month_number,
    d.month_name,
    COUNT(*) AS transaction_count,
    SUM(f.amount) AS total_amount,
    AVG(f.amount) AS average_transaction_amount
FROM adbdevbankproject.gold.fact_transactions f
JOIN adbdevbankproject.gold.dim_date d
    ON f.date_key = d.date_key
GROUP BY
    d.year_number,
    d.month_number,
    d.month_name
ORDER BY
    d.year_number,
    d.month_number;

year_number,month_number,month_name,transaction_count,total_amount,average_transaction_amount
2010,1,January,101209,4372532.12,43.202997
2010,2,February,93470,4103170.24,43.898259
2010,3,March,103345,4539853.38,43.929105
2010,4,April,100169,4407951.33,44.005145
2010,5,May,104773,4610601.80,44.005629
2010,6,June,102677,4509874.37,43.922927
2010,7,July,106034,4660340.55,43.951379
2010,8,August,107547,4688090.24,43.591083
2010,9,September,103902,4540028.90,43.695298
2010,10,October,106150,4623129.08,43.552794


In [0]:
%sql
-- Gold Analytics: Transaction Overview
-- Purpose: Understand the main measures and time range

SELECT
    COUNT(*) AS total_transactions,
    SUM(amount) AS total_amount,
    AVG(amount) AS average_transaction_amount,
    MIN(transaction_date) AS first_transaction_date,
    MAX(transaction_date) AS last_transaction_date
FROM adbdevbankproject.gold.fact_transactions;

total_transactions,total_amount,average_transaction_amount,first_transaction_date,last_transaction_date
13305909,571835398.56,42.976049,2010-01-01,2019-10-31


In [0]:
%sql
-- Final Gold table row counts

SELECT 'fact_transactions' AS table_name,
       COUNT(*) AS row_count
FROM adbdevbankproject.gold.fact_transactions

UNION ALL

SELECT 'dim_users',
       COUNT(*)
FROM adbdevbankproject.gold.dim_users

UNION ALL

SELECT 'dim_cards',
       COUNT(*)
FROM adbdevbankproject.gold.dim_cards

UNION ALL

SELECT 'dim_mcc',
       COUNT(*)
FROM adbdevbankproject.gold.dim_mcc

UNION ALL

SELECT 'dim_date',
       COUNT(*)
FROM adbdevbankproject.gold.dim_date;

table_name,row_count
fact_transactions,13305909
dim_users,2000
dim_cards,6146
dim_mcc,109
dim_date,3591
